# NVIDIA Corporation (NVDA) — Comprehensive Valuation Analysis

**Built with the tech-company-valuer framework · Feb 2026**

---

## Overview
NVIDIA is the dominant supplier of AI training and inference hardware (GPUs), holding ~80% of the AI accelerator market. Its CUDA software ecosystem creates a deep moat. This notebook models NVIDIA across 5 business segments using a segment-level DCF, Monte Carlo simulation, technical analysis, and peer comparison.

### Sections
1. Imports & Configuration
2. Live Data (yfinance)
3. Company Overview Dashboard
4. Segment Revenue Model
5. Technical Analysis
6. Segment-Level DCF Valuation
7. Sensitivity Heatmaps
8. Monte Carlo Simulation
9. Peer Comparison
10. FCF Waterfall Bridge
11. Investment Thesis

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 0: IMPORTS & NVIDIA CONFIGURATION
#  NVIDIA Corporation (NVDA) — FY2025 base (fiscal year ended Jan 2025)
#  Edit the variables below to customise assumptions
# ═══════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── Attempt yfinance import ───────────────────────────────────────
try:
    import yfinance as yf
    YF_AVAILABLE = True
except ImportError:
    print('⚠️  yfinance not installed. Run: pip install yfinance')
    YF_AVAILABLE = False

# ── COMPANY ──────────────────────────────────────────────────────
TICKER          = 'NVDA'
COMPANY_NAME    = 'NVIDIA Corporation'

# ── Market data (will be overwritten by yfinance if available) ───
SHARES_OUT_M    = 24_400    # million shares outstanding (diluted)
CURRENT_PRICE   = 115.0     # USD/share (approx at plan date)
MARKET_CAP_B    = 2_810.0   # USD billions
NET_DEBT_USD_M  = -38_000   # net cash position (NVDA has large cash pile)
BETA            = 1.7       # elevated AI-cycle beta

# ── FINANCIALS (FY2025, USD millions) — overwritten by yfinance ──
LATEST_REVENUE      = 130_497   # total revenue FY2025
LATEST_GROSS_PROFIT = 97_860    # gross profit FY2025
LATEST_OP_INCOME    = 81_453    # operating income FY2025
LATEST_NET_INCOME   = 72_880    # net income FY2025
LATEST_FCF          = 60_850    # free cash flow FY2025 (approx)
LATEST_SBC          =  4_500    # stock-based compensation FY2025
LATEST_CAPEX        =  3_200    # capital expenditure FY2025
LATEST_DA           =  1_400    # D&A FY2025

# ── REVENUE GROWTH ASSUMPTIONS (annual %) ────────────────────────
REVENUE_GROWTH_BEAR  = 0.12
REVENUE_GROWTH_BASE  = 0.30   # ← primary assumption; reflects AI tailwinds
REVENUE_GROWTH_BULL  = 0.55

# ── MARGIN ASSUMPTIONS ───────────────────────────────────────────
OPERATING_MARGIN_TERMINAL = 0.52  # NVDA CUDA moat → sustainably high margins
FCF_CONVERSION             = 0.90  # FCF as % of operating income at maturity

# ── DISCOUNT RATE ────────────────────────────────────────────────
RISK_FREE_RATE    = 0.043   # 10-yr Treasury ~4.3%
EQUITY_RISK_PREM  = 0.055   # Damodaran ERP estimate
COST_OF_DEBT      = 0.040
DEBT_WEIGHT       = 0.02    # NVDA is nearly debt-free (net cash)
TAX_RATE          = 0.13    # effective rate ~13% (international structure)

WACC_OVERRIDE     = None    # set a float to override WACC calc

# ── DCF HORIZON ──────────────────────────────────────────────────
FORECAST_YEARS      = 10
TERMINAL_GROWTH     = 0.035
FADE_GROWTH_TO      = 0.05   # revenue growth fades to this by final year

# ── MONTE CARLO ──────────────────────────────────────────────────
MC_SIMULATIONS      = 10_000
MC_REVENUE_STDEV    = 0.12   # higher uncertainty due to AI cycle
MC_MARGIN_STDEV     = 0.04
MC_WACC_STDEV       = 0.015

# ── SHARE-BASED COMPENSATION ─────────────────────────────────────
SBC_PCT_REVENUE     = 0.06   # ~6% of revenue (SBC/Rev)
ANNUAL_DILUTION_PCT = 0.005  # net dilution ~0.5% (buybacks offset SBC)

# ── CORPORATE G&A (USD millions per year, above segment opex) ────
CORPORATE_GA_USD_M  = 1_000

print('✅ Config loaded. Edit variables above and re-run to update all outputs.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 1: FETCH LIVE DATA FROM YFINANCE
# ═══════════════════════════════════════════════════════════════════

if YF_AVAILABLE:
    stock = yf.Ticker(TICKER)
    info  = stock.info

    COMPANY_NAME  = info.get('longName', COMPANY_NAME)
    CURRENT_PRICE = info.get('currentPrice', info.get('regularMarketPrice', CURRENT_PRICE))
    MARKET_CAP_B  = info.get('marketCap', 0) / 1e9
    SHARES_OUT_M  = info.get('sharesOutstanding', SHARES_OUT_M * 1e6) / 1e6
    BETA          = info.get('beta', BETA)

    total_debt   = info.get('totalDebt', 0) or 0
    total_cash   = info.get('totalCash', 0) or 0
    NET_DEBT_USD_M = (total_debt - total_cash) / 1e6

    income_stmt   = stock.income_stmt
    balance_sheet = stock.balance_sheet
    cash_flow     = stock.cash_flow
    quarterly_inc = stock.quarterly_income_stmt

    price_data = stock.history(period='2y')

    try:
        analyst_targets = {
            'target_mean': info.get('targetMeanPrice'),
            'target_low':  info.get('targetLowPrice'),
            'target_high': info.get('targetHighPrice'),
            'num_analysts': info.get('numberOfAnalystOpinions'),
            'recommendation': info.get('recommendationKey'),
        }
    except:
        analyst_targets = {}

    try:
        insider_txns = stock.insider_transactions
    except:
        insider_txns = pd.DataFrame()

    try:
        inst_holders = stock.institutional_holders
    except:
        inst_holders = pd.DataFrame()

    def safe_get(df, label, default=0):
        """Safely extract most recent value from a financial statement row."""
        if df is None or df.empty:
            return default
        if label in df.index:
            val = df.loc[label].dropna()
            if len(val) > 0:
                return float(val.iloc[0])
        return default

    LATEST_REVENUE      = safe_get(income_stmt, 'Total Revenue', LATEST_REVENUE * 1e6) / 1e6
    LATEST_GROSS_PROFIT = safe_get(income_stmt, 'Gross Profit',  LATEST_GROSS_PROFIT * 1e6) / 1e6
    LATEST_OP_INCOME    = safe_get(income_stmt, 'Operating Income', LATEST_OP_INCOME * 1e6) / 1e6
    LATEST_NET_INCOME   = safe_get(income_stmt, 'Net Income', LATEST_NET_INCOME * 1e6) / 1e6
    LATEST_FCF          = (safe_get(cash_flow, 'Operating Cash Flow', 0) -
                           abs(safe_get(cash_flow, 'Capital Expenditure', 0))) / 1e6
    if LATEST_FCF == 0:
        LATEST_FCF = 60_850
    LATEST_SBC   = safe_get(cash_flow, 'Stock Based Compensation', LATEST_SBC * 1e6) / 1e6
    LATEST_CAPEX = abs(safe_get(cash_flow, 'Capital Expenditure', LATEST_CAPEX * 1e6)) / 1e6
    LATEST_DA    = safe_get(cash_flow, 'Depreciation And Amortization', LATEST_DA * 1e6) / 1e6

    tax_provision = safe_get(income_stmt, 'Tax Provision')
    pretax_income = safe_get(income_stmt, 'Pretax Income')
    if pretax_income > 0 and tax_provision > 0:
        TAX_RATE = min(tax_provision / pretax_income, 0.30)

    if LATEST_REVENUE > 0 and LATEST_SBC > 0:
        SBC_PCT_REVENUE = LATEST_SBC / LATEST_REVENUE

    cost_of_equity = RISK_FREE_RATE + BETA * EQUITY_RISK_PREM
    equity_weight  = 1 - DEBT_WEIGHT
    WACC_CALC = (equity_weight * cost_of_equity +
                 DEBT_WEIGHT * COST_OF_DEBT * (1 - TAX_RATE))
    WACC = WACC_OVERRIDE if WACC_OVERRIDE else WACC_CALC

    print(f'📊 {COMPANY_NAME} ({TICKER})')
    print(f'   Price: ${CURRENT_PRICE:,.2f}  |  Mkt Cap: ${MARKET_CAP_B:,.1f}B  |  Beta: {BETA:.2f}')
    print(f'   Revenue: ${LATEST_REVENUE:,.0f}M  |  Op Income: ${LATEST_OP_INCOME:,.0f}M  |  FCF: ${LATEST_FCF:,.0f}M')
    print(f'   WACC: {WACC:.1%}  |  Tax Rate: {TAX_RATE:.1%}  |  SBC/Rev: {SBC_PCT_REVENUE:.1%}')
    print(f'   Net Debt: ${NET_DEBT_USD_M:,.0f}M  |  Shares: {SHARES_OUT_M:,.0f}M')
    if analyst_targets.get('target_mean'):
        print(f'   Analyst Target (mean): ${analyst_targets["target_mean"]:,.2f}  |  Rec: {analyst_targets.get("recommendation", "N/A").upper()}')
else:
    print('⚠️  yfinance not available — using manual FY2025 config values')
    cost_of_equity = RISK_FREE_RATE + BETA * EQUITY_RISK_PREM
    WACC = WACC_OVERRIDE or (cost_of_equity * (1 - DEBT_WEIGHT) +
                              COST_OF_DEBT * DEBT_WEIGHT * (1 - TAX_RATE))
    price_data = None
    income_stmt = None
    cash_flow = None
    balance_sheet = None
    analyst_targets = {}
    print(f'   Fallback WACC: {WACC:.1%}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 2: COMPANY OVERVIEW DASHBOARD
# ═══════════════════════════════════════════════════════════════════

if YF_AVAILABLE and income_stmt is not None and not income_stmt.empty:
    years = income_stmt.columns[:5]
    summary_data = []
    for yr in years:
        rev = income_stmt.loc['Total Revenue', yr]    / 1e9 if 'Total Revenue'    in income_stmt.index else 0
        gp  = income_stmt.loc['Gross Profit', yr]     / 1e9 if 'Gross Profit'     in income_stmt.index else 0
        op  = income_stmt.loc['Operating Income', yr] / 1e9 if 'Operating Income' in income_stmt.index else 0
        ni  = income_stmt.loc['Net Income', yr]       / 1e9 if 'Net Income'       in income_stmt.index else 0
        summary_data.append({
            'Year':              yr.strftime('%Y') if hasattr(yr, 'strftime') else str(yr),
            'Revenue ($B)':      f'{rev:,.1f}',
            'Gross Profit ($B)': f'{gp:,.1f}',
            'Gross Margin':      f'{gp/rev:.1%}' if rev > 0 else 'N/A',
            'Op Income ($B)':    f'{op:,.1f}',
            'Op Margin':         f'{op/rev:.1%}' if rev > 0 else 'N/A',
            'Net Income ($B)':   f'{ni:,.1f}',
            'Net Margin':        f'{ni/rev:.1%}' if rev > 0 else 'N/A',
        })
    df_summary = pd.DataFrame(summary_data)
    print('═' * 90)
    print(f'  {COMPANY_NAME} — Financial Summary')
    print('═' * 90)
    print(df_summary.to_string(index=False))

    # ── Revenue & Margin Trends Chart ─────────────────────────────
    years_plot, revenues, gross_margins, op_margins, net_margins = [], [], [], [], []
    for yr in reversed(list(income_stmt.columns[:5])):
        rev = income_stmt.loc['Total Revenue', yr]    / 1e9 if 'Total Revenue'    in income_stmt.index else 0
        gp  = income_stmt.loc['Gross Profit', yr]     / 1e9 if 'Gross Profit'     in income_stmt.index else 0
        op  = income_stmt.loc['Operating Income', yr] / 1e9 if 'Operating Income' in income_stmt.index else 0
        ni  = income_stmt.loc['Net Income', yr]       / 1e9 if 'Net Income'       in income_stmt.index else 0
        yr_str = yr.strftime('%Y') if hasattr(yr, 'strftime') else str(yr)
        years_plot.append(yr_str)
        revenues.append(rev)
        gross_margins.append(gp / rev if rev > 0 else 0)
        op_margins.append(op / rev if rev > 0 else 0)
        net_margins.append(ni / rev if rev > 0 else 0)

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=years_plot, y=revenues, name='Revenue ($B)',
                         marker_color='#76b900', opacity=0.75), secondary_y=False)
    fig.add_trace(go.Scatter(x=years_plot, y=gross_margins, name='Gross Margin',
                             line=dict(color='#4CAF50', width=2), mode='lines+markers'), secondary_y=True)
    fig.add_trace(go.Scatter(x=years_plot, y=op_margins, name='Op Margin',
                             line=dict(color='#FF9800', width=2), mode='lines+markers'), secondary_y=True)
    fig.add_trace(go.Scatter(x=years_plot, y=net_margins, name='Net Margin',
                             line=dict(color='#9C27B0', width=2), mode='lines+markers'), secondary_y=True)
    fig.update_layout(title=f'{COMPANY_NAME} — Revenue & Margin Trends',
                      template='plotly_white', hovermode='x unified',
                      legend=dict(orientation='h', y=-0.2))
    fig.update_yaxes(title_text='Revenue ($B)', secondary_y=False)
    fig.update_yaxes(title_text='Margin %', tickformat='.0%', secondary_y=True)
    fig.show()
else:
    print('ℹ️  Displaying FY2025 snapshot (yfinance data not available).')
    snapshot = {
        'Metric': ['Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 'FCF'],
        'FY2025 ($M)': [f'{LATEST_REVENUE:,.0f}', f'{LATEST_GROSS_PROFIT:,.0f}',
                        f'{LATEST_OP_INCOME:,.0f}', f'{LATEST_NET_INCOME:,.0f}',
                        f'{LATEST_FCF:,.0f}'],
        'Margin': ['-', f'{LATEST_GROSS_PROFIT/LATEST_REVENUE:.1%}',
                   f'{LATEST_OP_INCOME/LATEST_REVENUE:.1%}',
                   f'{LATEST_NET_INCOME/LATEST_REVENUE:.1%}',
                   f'{LATEST_FCF/LATEST_REVENUE:.1%}']
    }
    print(pd.DataFrame(snapshot).to_string(index=False))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 3: SEGMENT REVENUE MODEL
#  NVIDIA FY2025 segment revenues (USD millions) from 10-K
# ═══════════════════════════════════════════════════════════════════

segments = {
    'Data Center': {
        'current_revenue_usdm':      115_187,
        'growth_rate_bear':            0.15,
        'growth_rate_base':            0.35,   # AI infrastructure supercycle
        'growth_rate_bull':            0.60,
        'fade_to':                     0.05,
        'operating_margin_current':    0.65,   # very high due to pricing power
        'operating_margin_terminal':   0.55,   # some compression as competition grows
        'tam_2024_usdm':             100_000,
        'tam_2030_usdm':             500_000,
        'market_share_pct':             55,
        'key_driver': 'H100/H200/Blackwell GPU demand; hyperscaler AI capex',
    },
    'Gaming': {
        'current_revenue_usdm':       11_442,
        'growth_rate_bear':            0.03,
        'growth_rate_base':            0.10,
        'growth_rate_bull':            0.20,
        'fade_to':                     0.03,
        'operating_margin_current':    0.35,
        'operating_margin_terminal':   0.38,
        'tam_2024_usdm':              50_000,
        'tam_2030_usdm':              80_000,
        'market_share_pct':             82,    # GeForce dominant in discrete GPU
        'key_driver': 'GeForce RTX upgrade cycle; AI-enhanced gaming features',
    },
    'Professional Visualization': {
        'current_revenue_usdm':        1_688,
        'growth_rate_bear':            0.05,
        'growth_rate_base':            0.12,
        'growth_rate_bull':            0.22,
        'fade_to':                     0.04,
        'operating_margin_current':    0.40,
        'operating_margin_terminal':   0.45,
        'tam_2024_usdm':              15_000,
        'tam_2030_usdm':              35_000,
        'market_share_pct':             90,
        'key_driver': 'Omniverse, RTX workstations, generative AI tools',
    },
    'Automotive': {
        'current_revenue_usdm':        1_695,
        'growth_rate_bear':            0.20,
        'growth_rate_base':            0.45,
        'growth_rate_bull':            0.80,
        'fade_to':                     0.06,
        'operating_margin_current':    0.20,
        'operating_margin_terminal':   0.35,
        'tam_2024_usdm':              12_000,
        'tam_2030_usdm':              80_000,
        'market_share_pct':             12,
        'key_driver': 'DRIVE Orin/Thor platform; autonomous vehicle design wins',
    },
    'OEM & Other': {
        'current_revenue_usdm':          609,
        'growth_rate_bear':           -0.05,
        'growth_rate_base':            0.02,
        'growth_rate_bull':            0.08,
        'fade_to':                     0.02,
        'operating_margin_current':    0.25,
        'operating_margin_terminal':   0.25,
        'tam_2024_usdm':               5_000,
        'tam_2030_usdm':               5_000,
        'market_share_pct':             20,
        'key_driver': 'Legacy OEM channel — mature/declining segment',
    },
}

def project_segment(seg_name, seg, scenario='base', years=FORECAST_YEARS):
    """Project segment revenue with linearly fading growth rate."""
    growth_key     = f'growth_rate_{scenario}'
    initial_growth = seg[growth_key]
    terminal_growth = seg['fade_to']
    rows = []
    revenue        = seg['current_revenue_usdm']
    margin_current  = seg['operating_margin_current']
    margin_terminal = seg['operating_margin_terminal']
    for yr_offset in range(years):
        fade_frac = yr_offset / max(years - 1, 1)
        growth = initial_growth * (1 - fade_frac) + terminal_growth * fade_frac
        margin = margin_current * (1 - fade_frac) + margin_terminal * fade_frac
        revenue *= (1 + growth)
        op_income = revenue * margin
        rows.append({
            'year': datetime.now().year + yr_offset + 1,
            'segment': seg_name,
            'revenue_usdm': revenue,
            'growth_rate': growth,
            'op_margin': margin,
            'op_income_usdm': op_income,
        })
    return pd.DataFrame(rows)

projections = {}
for scenario in ['bear', 'base', 'bull']:
    dfs = [project_segment(name, seg, scenario) for name, seg in segments.items()]
    projections[scenario] = pd.concat(dfs, ignore_index=True)

# ── TAM summary ───────────────────────────────────────────────────
print('═' * 80)
print('  NVIDIA Segment Overview')
print('═' * 80)
for name, s in segments.items():
    rev = s['current_revenue_usdm']
    pct = rev / LATEST_REVENUE * 100
    print(f'  {name:<30} ${rev:>9,.0f}M  ({pct:.1f}% of rev)  |  {s["key_driver"]}')

# ── Stacked revenue projection chart (base case) ─────────────────
df_base  = projections['base']
df_bear  = projections['bear']
df_bull  = projections['bull']
fig = go.Figure()
colors = px.colors.qualitative.Set2
for i, seg_name in enumerate(segments.keys()):
    seg_data = df_base[df_base['segment'] == seg_name]
    fig.add_trace(go.Bar(x=seg_data['year'], y=seg_data['revenue_usdm'] / 1000,
                         name=seg_name, marker_color=colors[i % len(colors)]))
fig.update_layout(
    title=f'{COMPANY_NAME} — Projected Revenue by Segment (Base Case, $B)',
    xaxis_title='Year', yaxis_title='Revenue ($B)',
    barmode='stack', template='plotly_white',
    legend=dict(orientation='h', y=-0.2)
)
fig.show()

# ── Scenario total revenue comparison ────────────────────────────
fig2 = go.Figure()
scenario_colors = {'bear': '#F44336', 'base': '#2196F3', 'bull': '#4CAF50'}
for scen, color in scenario_colors.items():
    df_s = projections[scen].groupby('year')['revenue_usdm'].sum().reset_index()
    fig2.add_trace(go.Scatter(x=df_s['year'], y=df_s['revenue_usdm'] / 1000,
                              name=f'{scen.title()} Case',
                              line=dict(color=color, width=2.5)))
fig2.update_layout(title=f'{COMPANY_NAME} — Total Revenue Scenarios ($B)',
                   xaxis_title='Year', yaxis_title='Total Revenue ($B)',
                   template='plotly_white', legend=dict(orientation='h', y=-0.2))
fig2.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 4: TECHNICAL ANALYSIS
#  Candlestick + EMA50/200 + Bollinger Bands + MACD + RSI
# ═══════════════════════════════════════════════════════════════════

if YF_AVAILABLE and price_data is not None and len(price_data) > 50:
    df_ta = price_data.copy()

    df_ta['EMA_50']  = df_ta['Close'].ewm(span=50,  adjust=False).mean()
    df_ta['EMA_200'] = df_ta['Close'].ewm(span=200, adjust=False).mean()

    ema_12 = df_ta['Close'].ewm(span=12, adjust=False).mean()
    ema_26 = df_ta['Close'].ewm(span=26, adjust=False).mean()
    df_ta['MACD']        = ema_12 - ema_26
    df_ta['MACD_Signal'] = df_ta['MACD'].ewm(span=9, adjust=False).mean()
    df_ta['MACD_Hist']   = df_ta['MACD'] - df_ta['MACD_Signal']

    delta = df_ta['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    df_ta['RSI'] = 100 - (100 / (1 + rs))

    df_ta['BB_Mid']   = df_ta['Close'].rolling(20).mean()
    bb_std             = df_ta['Close'].rolling(20).std()
    df_ta['BB_Upper'] = df_ta['BB_Mid'] + 2 * bb_std
    df_ta['BB_Lower'] = df_ta['BB_Mid'] - 2 * bb_std

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03,
                        row_heights=[0.50, 0.15, 0.15, 0.20],
                        subplot_titles=['Price + EMAs + Bollinger', 'Volume', 'MACD', 'RSI (14)'])

    # Panel 1: Candlestick + EMAs + Bollinger
    fig.add_trace(go.Candlestick(x=df_ta.index,
                                  open=df_ta['Open'], high=df_ta['High'],
                                  low=df_ta['Low'],   close=df_ta['Close'],
                                  name='OHLC', showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['EMA_50'],  name='EMA 50',
                             line=dict(color='orange', width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['EMA_200'], name='EMA 200',
                             line=dict(color='red', width=1.2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['BB_Upper'], name='BB Upper',
                             line=dict(color='gray', width=0.5, dash='dot'), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['BB_Lower'], name='BB Lower',
                             line=dict(color='gray', width=0.5, dash='dot'), fill='tonexty',
                             fillcolor='rgba(128,128,128,0.08)', showlegend=False), row=1, col=1)

    # Panel 2: Volume
    colors_vol = ['#4CAF50' if c >= o else '#F44336'
                  for c, o in zip(df_ta['Close'], df_ta['Open'])]
    fig.add_trace(go.Bar(x=df_ta.index, y=df_ta['Volume'], name='Volume',
                         marker_color=colors_vol, showlegend=False), row=2, col=1)

    # Panel 3: MACD
    macd_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in df_ta['MACD_Hist']]
    fig.add_trace(go.Bar(x=df_ta.index, y=df_ta['MACD_Hist'], name='MACD Hist',
                         marker_color=macd_colors, showlegend=False), row=3, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['MACD'], name='MACD',
                             line=dict(color='#2196F3', width=1)), row=3, col=1)
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['MACD_Signal'], name='Signal',
                             line=dict(color='#FF9800', width=1)), row=3, col=1)

    # Panel 4: RSI
    fig.add_trace(go.Scatter(x=df_ta.index, y=df_ta['RSI'], name='RSI',
                             line=dict(color='#9C27B0', width=1.5)), row=4, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red',   row=4, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=4, col=1)

    fig.update_layout(
        title=f'{COMPANY_NAME} ({TICKER}) — Technical Analysis (2-Year)',
        template='plotly_white', height=900,
        legend=dict(orientation='h', y=1.02),
        xaxis_rangeslider_visible=False
    )
    fig.show()
else:
    print('⚠️  Price data not available — skipping technical analysis chart.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 5: SEGMENT-LEVEL DCF VALUATION
# ═══════════════════════════════════════════════════════════════════

def run_dcf(scenario='base', wacc_override=None, tg_override=None):
    """
    Segment-level DCF with SBC deduction and dilution adjustment.
    Returns dict with EV, equity value, fair value per share, and details.
    """
    wacc = wacc_override or WACC
    tg   = tg_override   or TERMINAL_GROWTH

    df_proj = projections[scenario]
    years   = sorted(df_proj['year'].unique())

    annual_fcfs    = []
    annual_details = []

    da_pct    = LATEST_DA    / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.01
    capex_pct = LATEST_CAPEX / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.03
    wc_pct    = 0.02

    for yr in years:
        yr_data        = df_proj[df_proj['year'] == yr]
        total_revenue  = yr_data['revenue_usdm'].sum()
        total_op_income = yr_data['op_income_usdm'].sum()

        sbc          = total_revenue * SBC_PCT_REVENUE
        corporate_ga = CORPORATE_GA_USD_M
        ebit         = total_op_income - corporate_ga - sbc
        taxes        = max(0, ebit * TAX_RATE)
        nopat        = ebit - taxes

        da       = total_revenue * da_pct
        capex    = total_revenue * capex_pct
        wc_chg   = total_revenue * wc_pct
        fcf      = nopat + da - capex - wc_chg

        annual_fcfs.append(fcf)
        annual_details.append({
            'year': yr, 'revenue': total_revenue, 'op_income': total_op_income,
            'sbc': sbc, 'ebit': ebit, 'nopat': nopat, 'da': da,
            'capex': capex, 'fcf': fcf
        })

    discount_factors = [(1 + wacc) ** -(i + 1) for i in range(len(annual_fcfs))]
    pv_fcfs          = np.array(annual_fcfs) * np.array(discount_factors)

    terminal_fcf   = annual_fcfs[-1] * (1 + tg)
    terminal_value = terminal_fcf / (wacc - tg)
    pv_terminal    = terminal_value * discount_factors[-1]

    enterprise_value = sum(pv_fcfs) + pv_terminal
    equity_value     = enterprise_value - NET_DEBT_USD_M

    diluted_shares       = SHARES_OUT_M * (1 + ANNUAL_DILUTION_PCT) ** FORECAST_YEARS
    fair_value_per_share = equity_value / diluted_shares if diluted_shares > 0 else 0
    upside               = (fair_value_per_share / CURRENT_PRICE - 1) if CURRENT_PRICE > 0 else 0

    terminal_revenue = annual_details[-1]['revenue']
    implied_pe       = enterprise_value / annual_details[-1]['nopat']  if annual_details[-1]['nopat']  > 0 else 0
    implied_ev_rev   = enterprise_value / terminal_revenue              if terminal_revenue > 0 else 0

    return {
        'scenario':            scenario,
        'ev_usdm':             round(enterprise_value, 0),
        'equity_value_usdm':   round(equity_value, 0),
        'fair_value_per_share':round(fair_value_per_share, 2),
        'upside_pct':          round(upside * 100, 1),
        'pv_fcfs_total':       round(sum(pv_fcfs), 0),
        'pv_terminal':         round(pv_terminal, 0),
        'terminal_pct_of_ev':  round(pv_terminal / enterprise_value * 100, 1) if enterprise_value > 0 else 0,
        'implied_pe':          round(implied_pe, 1),
        'implied_ev_rev':      round(implied_ev_rev, 1),
        'annual_details':      pd.DataFrame(annual_details),
        'pv_fcfs':             pv_fcfs,
        'wacc_used':           wacc,
    }

results = {
    'bear': run_dcf('bear'),
    'base': run_dcf('base'),
    'bull': run_dcf('bull'),
}

summary_rows = []
for label, r in results.items():
    summary_rows.append({
        'Scenario':          label.upper(),
        'EV ($B)':           f'${r["ev_usdm"]/1000:,.0f}',
        'Equity ($B)':       f'${r["equity_value_usdm"]/1000:,.0f}',
        'Fair Value / Share':f'${r["fair_value_per_share"]:,.2f}',
        'Upside/Downside':   f'{r["upside_pct"]:+.1f}%',
        'Terminal % of EV':  f'{r["terminal_pct_of_ev"]:.0f}%',
        'Implied Fwd P/E':   f'{r["implied_pe"]:.1f}x',
        'Implied EV/Rev':    f'{r["implied_ev_rev"]:.1f}x',
    })
df_scenarios = pd.DataFrame(summary_rows)
print('═' * 100)
print(f'  {COMPANY_NAME} — DCF Scenario Summary  (WACC: {WACC:.1%}, TGR: {TERMINAL_GROWTH:.1%})')
print(f'  Current Price: ${CURRENT_PRICE:,.2f}  |  Market Cap: ${MARKET_CAP_B:,.0f}B')
print('═' * 100)
print(df_scenarios.to_string(index=False))

# ── PV breakdown chart ────────────────────────────────────────────
base_r = results['base']
pv_years = list(range(1, FORECAST_YEARS + 1))
fig_pv = go.Figure()
fig_pv.add_trace(go.Bar(x=pv_years, y=base_r['pv_fcfs'],
                         name='PV of FCF', marker_color='#2196F3'))
fig_pv.add_hline(y=base_r['pv_terminal'] / FORECAST_YEARS,
                  line_dash='dash', line_color='orange',
                  annotation_text=f'PV Terminal: ${base_r["pv_terminal"]/1e6:.1f}T (avg/yr)')
fig_pv.update_layout(
    title=f'{COMPANY_NAME} — DCF: Present Value of FCFs by Year (Base Case)',
    xaxis_title='Forecast Year', yaxis_title='PV of FCF ($M)',
    template='plotly_white'
)
fig_pv.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 6: SENSITIVITY HEATMAPS
# ═══════════════════════════════════════════════════════════════════

# ── WACC vs Terminal Growth ───────────────────────────────────────
wacc_range = np.arange(0.08, 0.17, 0.01)
tg_range   = np.arange(0.02, 0.055, 0.005)

matrix = []
for w in wacc_range:
    row_vals = []
    for tg in tg_range:
        res = run_dcf('base', wacc_override=w, tg_override=tg)
        row_vals.append(res['fair_value_per_share'])
    matrix.append(row_vals)

fig_sens = px.imshow(
    matrix,
    text_auto='.0f',
    x=[f'{tg:.1%}' for tg in tg_range],
    y=[f'{w:.0%}'  for w  in wacc_range],
    color_continuous_scale='RdYlGn',
    title=f'{COMPANY_NAME} — Fair Value Sensitivity: WACC vs Terminal Growth Rate (Base Scenario)',
    labels=dict(x='Terminal Growth Rate', y='WACC', color='Fair Value ($)')
)
fig_sens.update_layout(template='plotly_white')
fig_sens.show()

# ── Revenue Growth vs Terminal Margin ─────────────────────────────
# Override segment growth proportionally
growth_mults = np.arange(0.5, 2.1, 0.25)   # 0.5x to 2.0x base growth
margin_range = np.arange(0.35, 0.65, 0.05)

# Temporarily modify global for this sensitivity
orig_margin = OPERATING_MARGIN_TERMINAL
matrix2 = []
for gm in growth_mults:
    row_vals2 = []
    # Build custom projections with scaled growth
    for tm in margin_range:
        custom_segs = {}
        for name, s in segments.items():
            cs = s.copy()
            cs['growth_rate_base']       = s['growth_rate_base'] * gm
            cs['operating_margin_terminal'] = tm
            custom_segs[name] = cs
        # Project
        custom_proj = pd.concat(
            [project_segment(n, s, 'base') for n, s in custom_segs.items()],
            ignore_index=True
        )
        # Quick DCF inline
        yrs = sorted(custom_proj['year'].unique())
        fcfs, dets = [], []
        for yr in yrs:
            yd = custom_proj[custom_proj['year'] == yr]
            rev = yd['revenue_usdm'].sum()
            opi = yd['op_income_usdm'].sum()
            sbc = rev * SBC_PCT_REVENUE
            eb  = opi - CORPORATE_GA_USD_M - sbc
            np_  = max(0, eb) * (1 - TAX_RATE)
            da_  = rev * (LATEST_DA / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.01)
            cx_  = rev * (LATEST_CAPEX / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.03)
            fcfs.append(np_ + da_ - cx_)
        disc = [(1 + WACC) ** -(i + 1) for i in range(len(fcfs))]
        pv   = sum(np.array(fcfs) * np.array(disc))
        tv   = fcfs[-1] * (1 + TERMINAL_GROWTH) / (WACC - TERMINAL_GROWTH) * disc[-1]
        ev   = pv + tv
        eq   = ev - NET_DEBT_USD_M
        sh   = SHARES_OUT_M * (1 + ANNUAL_DILUTION_PCT) ** FORECAST_YEARS
        fv   = eq / sh if sh > 0 else 0
        row_vals2.append(round(fv, 0))
    matrix2.append(row_vals2)

fig_sens2 = px.imshow(
    matrix2,
    text_auto='.0f',
    x=[f'{m:.0%}' for m in margin_range],
    y=[f'{g:.1f}x' for g in growth_mults],
    color_continuous_scale='RdYlGn',
    title=f'{COMPANY_NAME} — Fair Value Sensitivity: Growth Multiplier vs Terminal Op Margin (Base)',
    labels=dict(x='Terminal Op Margin', y='Growth vs Base', color='Fair Value ($)')
)
fig_sens2.update_layout(template='plotly_white')
fig_sens2.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 7: MONTE CARLO SIMULATION
#  10,000 simulations varying growth, margin, WACC, terminal growth
# ═══════════════════════════════════════════════════════════════════

np.random.seed(42)

def monte_carlo_dcf(n_sims=MC_SIMULATIONS):
    """Run MC simulation varying growth, margins, WACC, and terminal growth."""
    fair_values = []
    sim_params  = []

    for _ in range(n_sims):
        sim_growth = np.random.normal(REVENUE_GROWTH_BASE, MC_REVENUE_STDEV)
        sim_growth = np.clip(sim_growth, -0.05, 0.80)

        sim_margin = np.random.normal(OPERATING_MARGIN_TERMINAL, MC_MARGIN_STDEV)
        sim_margin = np.clip(sim_margin, 0.20, 0.75)

        sim_wacc   = np.random.normal(WACC, MC_WACC_STDEV)
        sim_wacc   = np.clip(sim_wacc, 0.07, 0.20)

        sim_tg     = np.random.uniform(0.02, 0.05)

        total_revenue = LATEST_REVENUE
        fcfs = []
        for yr in range(FORECAST_YEARS):
            fade_frac = yr / max(FORECAST_YEARS - 1, 1)
            growth    = sim_growth * (1 - fade_frac) + sim_tg * fade_frac
            total_revenue *= (1 + growth)
            sbc  = total_revenue * SBC_PCT_REVENUE
            ebit = total_revenue * sim_margin - CORPORATE_GA_USD_M - sbc
            nopat = ebit * (1 - TAX_RATE)
            da_   = total_revenue * (LATEST_DA    / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.01)
            cx_   = total_revenue * (LATEST_CAPEX / LATEST_REVENUE if LATEST_REVENUE > 0 else 0.03)
            fcfs.append(nopat + da_ - cx_)

        disc   = [(1 + sim_wacc) ** -(i + 1) for i in range(FORECAST_YEARS)]
        pv_fcf = sum(np.array(fcfs) * np.array(disc))

        if sim_wacc > sim_tg:
            tv    = fcfs[-1] * (1 + sim_tg) / (sim_wacc - sim_tg)
            pv_tv = tv * disc[-1]
        else:
            pv_tv = 0

        ev    = pv_fcf + pv_tv
        eq    = ev - NET_DEBT_USD_M
        sh    = SHARES_OUT_M * (1 + ANNUAL_DILUTION_PCT) ** FORECAST_YEARS
        fv    = eq / sh if sh > 0 else 0

        if 0 < fv < CURRENT_PRICE * 30:
            fair_values.append(fv)
            sim_params.append({'growth': sim_growth, 'margin': sim_margin,
                               'wacc': sim_wacc, 'tg': sim_tg, 'fv': fv})

    return np.array(fair_values), pd.DataFrame(sim_params)

mc_values, mc_params = monte_carlo_dcf()

# ── Histogram ─────────────────────────────────────────────────────
fig_mc = go.Figure()
fig_mc.add_trace(go.Histogram(x=mc_values, nbinsx=120, name='Simulated Fair Values',
                               marker_color='#76b900', opacity=0.75))
fig_mc.add_vline(x=CURRENT_PRICE, line_dash='dash', line_color='red',
                  annotation_text=f'Current: ${CURRENT_PRICE:,.0f}')
fig_mc.add_vline(x=np.median(mc_values), line_dash='dash', line_color='blue',
                  annotation_text=f'Median: ${np.median(mc_values):,.0f}')
fig_mc.update_layout(
    title=f'{COMPANY_NAME} — Monte Carlo Fair Value Distribution ({MC_SIMULATIONS:,} simulations)',
    xaxis_title='Fair Value per Share ($)', yaxis_title='Frequency',
    template='plotly_white'
)
fig_mc.show()

# ── Statistics ────────────────────────────────────────────────────
pct_upside = (mc_values > CURRENT_PRICE).mean() * 100
print(f'\n📊 Monte Carlo Results ({len(mc_values):,} valid simulations)')
print(f'   P(fair value > current ${CURRENT_PRICE:,.0f}): {pct_upside:.1f}%')
print(f'   10th percentile:  ${np.percentile(mc_values, 10):,.0f}')
print(f'   25th percentile:  ${np.percentile(mc_values, 25):,.0f}')
print(f'   Median:           ${np.median(mc_values):,.0f}')
print(f'   75th percentile:  ${np.percentile(mc_values, 75):,.0f}')
print(f'   90th percentile:  ${np.percentile(mc_values, 90):,.0f}')
print(f'   Mean:             ${np.mean(mc_values):,.0f}')
print(f'   Std Dev:          ${np.std(mc_values):,.0f}')

# ── CDF Chart ─────────────────────────────────────────────────────
sorted_vals = np.sort(mc_values)
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
fig_cdf = go.Figure()
fig_cdf.add_trace(go.Scatter(x=sorted_vals, y=cdf, mode='lines', name='CDF',
                              line=dict(color='#76b900', width=2)))
fig_cdf.add_vline(x=CURRENT_PRICE, line_dash='dash', line_color='red',
                   annotation_text='Current Price')
fig_cdf.update_layout(
    title=f'{COMPANY_NAME} — Cumulative Distribution of Fair Value',
    xaxis_title='Fair Value per Share ($)', yaxis_title='Cumulative Probability',
    template='plotly_white'
)
fig_cdf.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 8: PEER COMPARISON
#  AMD · INTC · AVGO · QCOM · TSM
# ═══════════════════════════════════════════════════════════════════

PEER_TICKERS = ['AMD', 'INTC', 'AVGO', 'QCOM', 'TSM']

if YF_AVAILABLE:
    peer_data = []
    for t in [TICKER] + PEER_TICKERS:
        try:
            p   = yf.Ticker(t)
            pi  = p.info
            p_inc = p.income_stmt
            p_cf  = p.cash_flow

            rev  = p_inc.loc['Total Revenue'].dropna().iloc[0]    / 1e9 if 'Total Revenue'    in p_inc.index else 0
            oi   = p_inc.loc['Operating Income'].dropna().iloc[0] / 1e9 if 'Operating Income' in p_inc.index else 0
            ni   = p_inc.loc['Net Income'].dropna().iloc[0]       / 1e9 if 'Net Income'       in p_inc.index else 0
            ocf  = p_cf.loc['Operating Cash Flow'].dropna().iloc[0] / 1e9 if 'Operating Cash Flow' in p_cf.index else 0
            cpx  = abs(p_cf.loc['Capital Expenditure'].dropna().iloc[0]) / 1e9 if 'Capital Expenditure' in p_cf.index else 0
            fcf_ = ocf - cpx

            mcap   = pi.get('marketCap',         0) / 1e9
            ev_val = pi.get('enterpriseValue',    0) / 1e9

            if 'Total Revenue' in p_inc.index and len(p_inc.loc['Total Revenue'].dropna()) >= 2:
                rv = p_inc.loc['Total Revenue'].dropna()
                rev_growth = (rv.iloc[0] / rv.iloc[1] - 1) if rv.iloc[1] != 0 else 0
            else:
                rev_growth = 0

            peer_data.append({
                'Ticker':        t,
                'Mkt Cap ($B)':  round(mcap, 1),
                'EV ($B)':       round(ev_val, 1),
                'Revenue ($B)':  round(rev, 1),
                'Rev Growth':    f'{rev_growth:.0%}',
                'Op Margin':     f'{oi/rev:.0%}' if rev > 0 else 'N/A',
                'Net Margin':    f'{ni/rev:.0%}' if rev > 0 else 'N/A',
                'FCF ($B)':      round(fcf_, 1),
                'P/E (trail)':   round(pi.get('trailingPE', 0), 1),
                'EV/EBITDA':     round(pi.get('enterpriseToEbitda', 0), 1),
                'EV/Rev':        round(ev_val / rev, 1) if rev > 0 else 0,
                'P/FCF':         round(mcap / fcf_, 1) if fcf_ > 0 else 0,
                '_rev_growth_num': rev_growth,
                '_ev_rev_num':   ev_val / rev if rev > 0 else 0,
                '_mcap_num':     mcap,
            })
        except Exception as e:
            print(f'⚠️  Could not fetch {t}: {e}')

    df_peers = pd.DataFrame(peer_data)
    print('═' * 110)
    print('  Semiconductor / AI Peer Comparison')
    print('═' * 110)
    display_cols = [c for c in df_peers.columns if not c.startswith('_')]
    print(df_peers[display_cols].to_string(index=False))

    # ── Bubble chart: EV/Revenue vs Revenue Growth ────────────────
    fig_peer = px.scatter(
        df_peers, x='_rev_growth_num', y='_ev_rev_num',
        size='_mcap_num', text='Ticker', color='Ticker',
        title='Peer Comparison: EV/Revenue vs Revenue Growth (bubble size = market cap)',
        labels={'_rev_growth_num': 'Revenue Growth (YoY)', '_ev_rev_num': 'EV/Revenue', '_mcap_num': 'Mkt Cap'}
    )
    fig_peer.update_traces(textposition='top center')
    fig_peer.update_layout(template='plotly_white', xaxis_tickformat='.0%',
                            showlegend=False)
    fig_peer.show()
else:
    print('⚠️  yfinance not available — skipping live peer comparison.')
    print('   Peers to compare: NVDA vs AMD, INTC, AVGO, QCOM, TSM')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  SECTION 9: FCF WATERFALL BRIDGE
#  Year 3 Base Case — Revenue → FCF
# ═══════════════════════════════════════════════════════════════════

base_details = results['base']['annual_details']
yr3 = base_details.iloc[2]   # Year 3 of 10-year projection

seg_opex_yr3 = yr3['revenue'] - yr3['op_income']

labels   = ['Revenue', 'Segment OpEx', 'SBC', 'Corp G&A', 'EBIT', 'Tax', 'NOPAT', 'D&A', 'Capex', 'ΔWC', 'FCF']
values   = [
    yr3['revenue'],
    -seg_opex_yr3,
    -yr3['sbc'],
    -CORPORATE_GA_USD_M,
    yr3['ebit'],
    -(yr3['ebit'] * TAX_RATE),
    yr3['nopat'],
    yr3['da'],
    -yr3['capex'],
    -(yr3['revenue'] * 0.02),    # working capital change estimate
    yr3['fcf'],
]
measures = ['absolute', 'relative', 'relative', 'relative', 'total',
            'relative', 'total', 'relative', 'relative', 'relative', 'total']

fig_wf = go.Figure(go.Waterfall(
    name='FCF Bridge', orientation='v',
    measure=measures, x=labels, y=values,
    connector={'line': {'color': 'rgb(63,63,63)'}},
    increasing={'marker': {'color': '#4CAF50'}},
    decreasing={'marker': {'color': '#F44336'}},
    totals={'marker': {'color': '#76b900'}}
))
fig_wf.update_layout(
    title=f'{COMPANY_NAME} — FCF Bridge: Base Case Year 3 of Projection ($M)',
    yaxis_title='USD Millions',
    template='plotly_white'
)
fig_wf.show()

print(f'\n  Year 3 Base Case Summary:')
print(f'   Revenue:      ${yr3["revenue"]:>12,.0f}M')
print(f'   EBIT:         ${yr3["ebit"]:>12,.0f}M  ({yr3["ebit"]/yr3["revenue"]:.1%} margin)')
print(f'   NOPAT:        ${yr3["nopat"]:>12,.0f}M')
print(f'   FCF:          ${yr3["fcf"]:>12,.0f}M  ({yr3["fcf"]/yr3["revenue"]:.1%} FCF margin)')

---

# Section 10 — NVIDIA Investment Thesis

## Business Overview

NVIDIA designs and sells GPUs, system-on-chip units, and AI software platforms. Its **Data Center** segment (H100/H200/Blackwell GPUs, NVLink networking, DGX systems) now constitutes ~88% of revenue. The company's **CUDA** software ecosystem — developed over 15+ years — creates switching costs that reinforce its hardware dominance.

---

## Scenario Summaries

### 🐂 Bull Case
- AI infrastructure supercycle continues through 2027+; Blackwell demand remains constrained by supply
- Sovereign AI, enterprise AI, and inference deployment all accelerate simultaneously
- NVIDIA expands into robotics (Isaac), agentic AI, and physical AI beyond the datacenter
- Automotive (DRIVE platform) contributes meaningfully by 2027–2028
- Data center revenue reaches $400B+ by 2030; consolidated margins hold ~55%+

### 📊 Base Case
- Data center growth moderates from 200%+ to 30–35% YoY as market normalises
- AMD MI300 series gains inference share but NVIDIA retains training dominance
- Custom silicon from hyperscalers (Google TPU, Amazon Trainium) displaces ~15–20% of potential NVDA revenue
- Gaming recovers modestly; automotive scales slowly
- WACC ~13%; terminal operating margin ~52%

### 🐻 Bear Case
- Hyperscaler capex slows materially (recession, ROI concerns on AI spend)
- Custom silicon and AMD gain faster-than-expected share in inference
- Export controls on China sales expand, closing a significant revenue segment
- Semiconductor cycle turns down; inventory correction hits pricing
- Operating margins compress to 40–45% as competition intensifies

---

## Key Risks

| Risk | Probability | Impact | Mitigation |
|------|-------------|--------|------------|
| China export controls escalate | Medium | High | Diversify customer base; develop export-compliant chips |
| Hyperscaler capex slowdown | Medium | Very High | Long-term contracts; multi-year design cycle |
| AMD / custom silicon share gains | High | Medium | CUDA moat; software depth; NVLink ecosystem |
| Semiconductor cyclicality | Medium | High | Diversification into automotive, edge AI |
| AI ROI concerns reduce demand | Low-Medium | Very High | Demonstrate productivity gains at enterprise level |
| Antitrust / regulatory scrutiny | Low | Medium | Proactive compliance; open standards advocacy |

---

## Key Catalysts

- **Blackwell ramp**: GB200/GB300 NVL72 rack-scale systems commanding $2–3M per rack
- **Inference monetisation**: Shift from training to inference expands addressable market
- **Sovereign AI**: Governments building national AI infrastructure (EU, India, Japan, Middle East)
- **Automotive revenue inflection**: DRIVE Thor production launches 2025–2027
- **CUDA ecosystem**: 4M+ developers; growing moat in software and tooling
- **AI agents & robotics**: Next platform cycle beyond LLMs

---

## Valuation Summary

| Method | Bear | Base | Bull |
|--------|------|------|------|
| Segment DCF (10yr, WACC ~13%) | See output | See output | See output |
| Monte Carlo (10th/Median/90th) | See output | See output | See output |

> **Analyst context** (if yfinance available): See Section 1 output for consensus target price and recommendation.

---

*This notebook was generated by the `tech-company-valuer` framework. All projections are forward-looking estimates and not investment advice. DYOR.*